In [1]:

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
mes = 'may_2026'

In [3]:
#Archivos 

cerrado = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/IRI_cerrrado.csv", delimiter=';')

etapa_1 = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/IRI_Etapa1.csv", delimiter=';')

turno_ccz = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_CCZ.csv",delimiter=';')

turno_via = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_Via.csv",delimiter=';')

turno_sup = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Turnos_Sup.csv",delimiter=';')

asistencia = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Asistencia.csv",encoding='latin', delimiter=';')

tipo_dia = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Tipo día.csv", encoding='latin',delimiter=';')

personal = pd.read_csv("Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/Personal_id.csv",encoding='latin',delimiter=';')

In [4]:
#consolidar iri con los datos de cerrado y etapa_1

iri = pd.concat([cerrado,etapa_1], ignore_index=True)

iri

,IdIRI,Estado,Fecha Viaje,F. Inicio DP,F. Cierre DP,Fuente,Servicio,IdViaje,ViajeLinea,Coche,...,Imputación de datos,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia
0,88477101,No Contestado,1/05/2026,5/05/2026 0:00,11/05/2026 23:59,ALIMENTACION,CO11D0001,2,1,2,...,Con hora de Inicio LG SMART OPERATOR,1,0,0.00,-0.07,104.000.000,1,0,1,NaN
1,88477102,No Contestado,1/05/2026,5/05/2026 0:00,11/05/2026 23:59,ALIMENTACION,CO11D0001,3,2,2,...,Con hora de Inicio LG SMART OPERATOR,0,0,12.88,-0.05,104.000.000,1,0,1,NaN
2,88477103,No Contestado,1/05/2026,5/05/2026 0:00,11/05/2026 23:59,ALIMENTACION,CO11D0001,4,3,2,...,Con hora de Inicio LG SMART OPERATOR,0,0,13.90,-0.13,104.000.000,1,0,1,NaN
3,88477104,No Contestado,1/05/2026,5/05/2026 0:00,11/05/2026 23:59,ALIMENTACION,CO11D0001,5,4,2,...,Con hora de Inicio LG SMART OPERATOR,0,0,14.12,0.05,150.000.000,1,0,1,NaN
4,88477105,No Contestado,1/05/2026,5/05/2026 0:00,11/05/2026 23:59,ALIMENTACION,CO11D0001,6,5,2,...,Con hora de Inicio LG SMART OPERATOR,0,0,13.05,-0.50,150.000.000,1,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,88657219,Validado,5/05/2026,7/05/2026 0:00,13/05/2026 23:59,URBANO,CE17BG021,1,1,29,...,Con hora de Inicio LG SMART OPERATOR,0,0,22.07,3.15,300.000.000,0,0,1,NaN
32614,88777726,Validado,8/05/2026,12/05/2026 0:00,19/05/2026 23:59,URBANO,CE17BG024,1,1,32,...,Con hora de Inicio LG SMART OPERATOR,0,0,31.85,12.20,200.000.000,0,0,1,NaN
32615,88695100,Validado,6/05/2026,8/05/2026 0:00,14/05/2026 23:59,URBANO,CE12D0036,4,3,36,...,Con hora de Inicio LG SMART OPERATOR,0,0,7.75,0.57,85.714.286,1,0,1,NaN
32616,88776799,Validado,8/05/2026,12/05/2026 0:00,19/05/2026 23:59,URBANO,CE1600008,2,1,8,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.65,-0.57,60.000.000,1,0,1,NaN


In [5]:
#Eliminar columnas 

col_eliminar = ['IdIRI', 'Estado', 'F. Inicio DP', 'F. Cierre DP','Operador Programado','Id Operador','Operador']

iri = iri.drop(columns=col_eliminar)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Imputación de datos,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,Con hora de Inicio LG SMART OPERATOR,1,0,0.00,-0.07,104.000.000,1,0,1,NaN
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,Con hora de Inicio LG SMART OPERATOR,0,0,12.88,-0.05,104.000.000,1,0,1,NaN
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,Con hora de Inicio LG SMART OPERATOR,0,0,13.90,-0.13,104.000.000,1,0,1,NaN
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,Con hora de Inicio LG SMART OPERATOR,0,0,14.12,0.05,150.000.000,1,0,1,NaN
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,Con hora de Inicio LG SMART OPERATOR,0,0,13.05,-0.50,150.000.000,1,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,Con hora de Inicio LG SMART OPERATOR,0,0,22.07,3.15,300.000.000,0,0,1,NaN
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,Con hora de Inicio LG SMART OPERATOR,0,0,31.85,12.20,200.000.000,0,0,1,NaN
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,Con hora de Inicio LG SMART OPERATOR,0,0,7.75,0.57,85.714.286,1,0,1,NaN
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,Con hora de Inicio LG SMART OPERATOR,0,0,5.65,-0.57,60.000.000,1,0,1,NaN


In [6]:
iri['Cantidad'] = 1

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Primer Viaje,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,1,0,0.00,-0.07,104.000.000,1,0,1,NaN,1
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,0,0,12.88,-0.05,104.000.000,1,0,1,NaN,1
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,0,0,13.90,-0.13,104.000.000,1,0,1,NaN,1
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,0,0,14.12,0.05,150.000.000,1,0,1,NaN,1
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,0,0,13.05,-0.50,150.000.000,1,0,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,0,0,22.07,3.15,300.000.000,0,0,1,NaN,1
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,0,0,31.85,12.20,200.000.000,0,0,1,NaN,1
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,0,0,7.75,0.57,85.714.286,1,0,1,NaN,1
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,0,0,5.65,-0.57,60.000.000,1,0,1,NaN,1


In [7]:
#Convertir datos de columna Hora Teórica

# Expresión regular para extraer la hora
patron_hora = r'(\d{1,2}:\d{2}:\d{2}\s[ap]\.?\s?[mM]\.)'

# Aplicar la extracción de hora a la columna 'Hora Teórica'
iri['Hora'] = iri['Hora Teórica'].str.extract(patron_hora)

# Mostrar el DataFrame resultante
iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Ultimo Viaje,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,0,0.00,-0.07,104.000.000,1,0,1,NaN,1,NaN
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,0,12.88,-0.05,104.000.000,1,0,1,NaN,1,NaN
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,0,13.90,-0.13,104.000.000,1,0,1,NaN,1,NaN
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,0,14.12,0.05,150.000.000,1,0,1,NaN,1,NaN
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,0,13.05,-0.50,150.000.000,1,0,1,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,0,22.07,3.15,300.000.000,0,0,1,NaN,1,NaN
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,0,31.85,12.20,200.000.000,0,0,1,NaN,1,NaN
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,0,7.75,0.57,85.714.286,1,0,1,NaN,1,NaN
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,0,5.65,-0.57,60.000.000,1,0,1,NaN,1,NaN


In [8]:
# Dividir la columna 'Hora Teórica' en 'Fecha' y 'Hora'
iri[['Fecha1', 'Hora']] = iri['Hora Teórica'].str.split(' ', n=1, expand=True)

# Mostrar el DataFrame resultante
iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Int Real de Paso Recal,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,0.00,-0.07,104.000.000,1,0,1,NaN,1,4:08,1/05/2026
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,12.88,-0.05,104.000.000,1,0,1,NaN,1,4:34,1/05/2026
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,13.90,-0.13,104.000.000,1,0,1,NaN,1,5:00,1/05/2026
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,14.12,0.05,150.000.000,1,0,1,NaN,1,5:26,1/05/2026
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,13.05,-0.50,150.000.000,1,0,1,NaN,1,5:52,1/05/2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,22.07,3.15,300.000.000,0,0,1,NaN,1,NaN,NaN
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,31.85,12.20,200.000.000,0,0,1,NaN,1,NaN,NaN
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,7.75,0.57,85.714.286,1,0,1,NaN,1,10:38,6/05/2026
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,5.65,-0.57,60.000.000,1,0,1,NaN,1,3:43,8/05/2026


In [9]:
# Dividir la columna  en partes usando ':', y seleccionar la primera parte (horas)
iri['franja'] = iri['Hora'].str.split(':').str[0]

# Convertir la columna 'franja' a tipo entero
iri['franja'] = pd.to_numeric(iri['franja'], errors='coerce').astype('Int64')

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Dif Hora Real y Teor,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,-0.07,104.000.000,1,0,1,NaN,1,4:08,1/05/2026,4
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,-0.05,104.000.000,1,0,1,NaN,1,4:34,1/05/2026,4
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,-0.13,104.000.000,1,0,1,NaN,1,5:00,1/05/2026,5
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,0.05,150.000.000,1,0,1,NaN,1,5:26,1/05/2026,5
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,-0.50,150.000.000,1,0,1,NaN,1,5:52,1/05/2026,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,3.15,300.000.000,0,0,1,NaN,1,NaN,NaN,<NA>
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,12.20,200.000.000,0,0,1,NaN,1,NaN,NaN,<NA>
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,0.57,85.714.286,1,0,1,NaN,1,10:38,6/05/2026,10
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,-0.57,60.000.000,1,0,1,NaN,1,3:43,8/05/2026,3


In [10]:
#Llevar ruta comercial a iri

def calcular_posicion(Linea):
    
    filtro = (
        (turno_via['Linea'] == Linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_via.loc[filtro].empty:
        # Obtener el primer valor
        ruta = turno_via.loc[filtro, 'Ruta'].iloc[0]
        return ruta  if not pd.isna(ruta) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Ruta_Comercial'] = iri.apply(
    lambda row: calcular_posicion(
        row['Linea SAE']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Intervalo Regulado,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,104.000.000,1,0,1,NaN,1,4:08,1/05/2026,4,1-ene
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,104.000.000,1,0,1,NaN,1,4:34,1/05/2026,4,1-ene
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,104.000.000,1,0,1,NaN,1,5:00,1/05/2026,5,1-ene
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,150.000.000,1,0,1,NaN,1,5:26,1/05/2026,5,1-ene
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,150.000.000,1,0,1,NaN,1,5:52,1/05/2026,5,1-ene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,300.000.000,0,0,1,NaN,1,NaN,NaN,<NA>,None
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,200.000.000,0,0,1,NaN,1,NaN,NaN,<NA>,None
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,85.714.286,1,0,1,NaN,1,10:38,6/05/2026,10,None
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,60.000.000,1,0,1,NaN,1,3:43,8/05/2026,3,None


In [11]:
#Llevar tipo día a iri

def calcular_posicion(fecha):
    
    filtro = (
        (tipo_dia['Fecha'] == fecha)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not tipo_dia.loc[filtro].empty:
        # Obtener el primer valor
        Tipo_día = tipo_dia.loc[filtro, 'Tipo Día'].iloc[0]
        return Tipo_día  if not pd.isna(Tipo_día ) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Tipo dia'] = iri.apply(
    lambda row: calcular_posicion(
        row['Fecha Viaje']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Viaje Regular,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,1,0,1,NaN,1,4:08,1/05/2026,4,1-ene,Festivo
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,1,0,1,NaN,1,4:34,1/05/2026,4,1-ene,Festivo
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,1,0,1,NaN,1,5:00,1/05/2026,5,1-ene,Festivo
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,1,0,1,NaN,1,5:26,1/05/2026,5,1-ene,Festivo
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,1,0,1,NaN,1,5:52,1/05/2026,5,1-ene,Festivo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,0,0,1,NaN,1,NaN,NaN,<NA>,None,Hábil
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,0,0,1,NaN,1,NaN,NaN,<NA>,None,Hábil
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,1,0,1,NaN,1,10:38,6/05/2026,10,None,Hábil
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,1,0,1,NaN,1,3:43,8/05/2026,3,None,Hábil


In [12]:
# Función para obtener la letra inicial de cada palabra
def obtener_letra_inicial(texto):
    palabras = texto.split()
    iniciales = [palabra[0] for palabra in palabras]
    return ''.join(iniciales)

# Aplicar la función a la columna 'Tipo día'
iri['Tipo_dia'] = iri['Tipo dia'].apply(obtener_letra_inicial)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Enviado en Vacío,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,0,1,NaN,1,4:08,1/05/2026,4,1-ene,Festivo,F
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,0,1,NaN,1,4:34,1/05/2026,4,1-ene,Festivo,F
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,0,1,NaN,1,5:00,1/05/2026,5,1-ene,Festivo,F
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,0,1,NaN,1,5:26,1/05/2026,5,1-ene,Festivo,F
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,0,1,NaN,1,5:52,1/05/2026,5,1-ene,Festivo,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,0,1,NaN,1,NaN,NaN,<NA>,None,Hábil,H
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,0,1,NaN,1,NaN,NaN,<NA>,None,Hábil,H
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,0,1,NaN,1,10:38,6/05/2026,10,None,Hábil,H
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,0,1,NaN,1,3:43,8/05/2026,3,None,Hábil,H


In [13]:
#Traer el turno de reg_via

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_via['Linea'] == Linea)&
        (turno_via['Ruta_SAE'] == Ruta_SAE)&
        (turno_via['Franja'] == Franja)&
        (turno_via['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_via.loc[filtro].empty:
        # Obtener el primer valor
        turno = turno_via.loc[filtro, 'Turno'].iloc[0]
        return turno if not pd.isna(turno) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_via'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Ruta Calcula IRI,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,1,NaN,1,4:08,1/05/2026,4,1-ene,Festivo,F,None
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,1,NaN,1,4:34,1/05/2026,4,1-ene,Festivo,F,None
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,1,NaN,1,5:00,1/05/2026,5,1-ene,Festivo,F,PLA3
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,1,NaN,1,5:26,1/05/2026,5,1-ene,Festivo,F,PLA3
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,1,NaN,1,5:52,1/05/2026,5,1-ene,Festivo,F,PLA3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,1,NaN,1,NaN,NaN,<NA>,None,Hábil,H,None
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,1,NaN,1,NaN,NaN,<NA>,None,Hábil,H,None
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,1,NaN,1,10:38,6/05/2026,10,None,Hábil,H,None
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,1,NaN,1,3:43,8/05/2026,3,None,Hábil,H,None


In [14]:
#Traer el turno de control

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_ccz['Linea'] == Linea)&
        (turno_ccz['Ruta_SAE'] == Ruta_SAE)&
        (turno_ccz['Franja'] == Franja)&
        (turno_ccz['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_ccz.loc[filtro].empty:
        # Obtener el primer valor
        turnos = turno_ccz.loc[filtro, 'Turno'].iloc[0]
        return turnos if not pd.isna(turnos) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_ccz'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Evidencia,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,NaN,1,4:08,1/05/2026,4,1-ene,Festivo,F,None,None
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,NaN,1,4:34,1/05/2026,4,1-ene,Festivo,F,None,None
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,NaN,1,5:00,1/05/2026,5,1-ene,Festivo,F,PLA3,None
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,NaN,1,5:26,1/05/2026,5,1-ene,Festivo,F,PLA3,None
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,NaN,1,5:52,1/05/2026,5,1-ene,Festivo,F,PLA3,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,NaN,1,NaN,NaN,<NA>,None,Hábil,H,None,None
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,NaN,1,NaN,NaN,<NA>,None,Hábil,H,None,None
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,NaN,1,10:38,6/05/2026,10,None,Hábil,H,None,None
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,NaN,1,3:43,8/05/2026,3,None,Hábil,H,None,None


In [15]:
#Traer el turno de supervisor

def calcular_turno(Linea,Ruta_SAE,Franja,Tipo_Dia):
    
    filtro = (
        (turno_sup['Linea'] == Linea)&
        (turno_sup['Ruta_SAE'] == Ruta_SAE)&
        (turno_sup['Franja'] == Franja)&
        (turno_sup['Tipo Dia'] == Tipo_Dia)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not turno_sup.loc[filtro].empty:
        # Obtener el primer valor
        turno = turno_sup.loc[filtro, 'Estacion'].iloc[0]
        return turno if not pd.isna(turno) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Turno_Sup'] = iri.apply(
    lambda row: calcular_turno(
        row['Linea SAE'],
        row['Ruta SAE'],
        row['franja'],
        row['Tipo_dia']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Cantidad,Hora,Fecha1,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,1,4:08,1/05/2026,4,1-ene,Festivo,F,None,None,SUP CCZ
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,1,4:34,1/05/2026,4,1-ene,Festivo,F,None,None,SUP CCZ
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,1,5:00,1/05/2026,5,1-ene,Festivo,F,PLA3,None,SUP CCZ
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,1,5:26,1/05/2026,5,1-ene,Festivo,F,PLA3,None,SUP CCZ
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,1,5:52,1/05/2026,5,1-ene,Festivo,F,PLA3,None,SUP CCZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,1,NaN,NaN,<NA>,None,Hábil,H,None,None,None
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,1,NaN,NaN,<NA>,None,Hábil,H,None,None,None
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,1,10:38,6/05/2026,10,None,Hábil,H,None,None,None
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,1,3:43,8/05/2026,3,None,Hábil,H,None,None,None


In [16]:
#Traer el regulador según el turno

def calcular_turno(Fecha,TURNO_Programado):
    
    filtro = (
        (asistencia['Fecha'] == Fecha)&
        (asistencia['TURNO Programado'] == TURNO_Programado)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not asistencia.loc[filtro].empty:
        # Obtener el primer valor
        asistencias = asistencia.loc[filtro, 'Nombre Completo'].iloc[0]
        return asistencias if not pd.isna(asistencias) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['reg_via'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_via']
    ),
    axis=1
)

iri['reg_ccz'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_ccz']
    ),
    axis=1
)

iri['Supervisor'] = iri.apply(
    lambda row: calcular_turno(
        row['Fecha Viaje'],
        row['Turno_Sup']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,franja,Ruta_Comercial,Tipo dia,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup,reg_via,reg_ccz,Supervisor
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,4,1-ene,Festivo,F,None,None,SUP CCZ,None,None,CAMILO ANDRES LOPEZ PENAGOS
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,4,1-ene,Festivo,F,None,None,SUP CCZ,None,None,CAMILO ANDRES LOPEZ PENAGOS
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,5,1-ene,Festivo,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,5,1-ene,Festivo,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,5,1-ene,Festivo,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,<NA>,None,Hábil,H,None,None,None,None,None,None
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,<NA>,None,Hábil,H,None,None,None,None,None,None
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,10,None,Hábil,H,None,None,None,None,None,None
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,3,None,Hábil,H,None,None,None,None,None,None


In [17]:
#Llevar identificación al regulador

def calcular_posicion(Nombre_completo):
    
    filtro = (
        (personal['Nombre Completo'] == Nombre_completo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not personal.loc[filtro].empty:
        # Obtener el primer valor
        personas = personal.loc[filtro, 'Identificación'].iloc[0]
        return  personas  if not pd.isna( personas ) else None  
    
    return 0  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
iri['Id_via'] = iri.apply(
    lambda row: calcular_posicion(
        row['reg_via']
    ),
    axis=1
)

iri['Id_ccz'] = iri.apply(
    lambda row: calcular_posicion(
        row['reg_ccz']
    ),
    axis=1
)

iri['Id_sup'] = iri.apply(
    lambda row: calcular_posicion(
        row['Supervisor']
    ),
    axis=1
)

iri

,Fecha Viaje,Fuente,Servicio,IdViaje,ViajeLinea,Coche,Id Operador Programado,Linea SAE,Ruta SAE,Vehiculo,...,Tipo_dia,Turno_via,Turno_ccz,Turno_Sup,reg_via,reg_ccz,Supervisor,Id_via,Id_ccz,Id_sup
0,1/05/2026,ALIMENTACION,CO11D0001,2,1,2,104,10504,10990,622,...,F,None,None,SUP CCZ,None,None,CAMILO ANDRES LOPEZ PENAGOS,0,0,1023870284
1,1/05/2026,ALIMENTACION,CO11D0001,3,2,2,104,10504,10990,622,...,F,None,None,SUP CCZ,None,None,CAMILO ANDRES LOPEZ PENAGOS,0,0,1023870284
2,1/05/2026,ALIMENTACION,CO11D0001,4,3,2,104,10504,10990,622,...,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS,1024494610,0,1023870284
3,1/05/2026,ALIMENTACION,CO11D0001,5,4,2,104,10504,10990,622,...,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS,1024494610,0,1023870284
4,1/05/2026,ALIMENTACION,CO11D0001,6,5,2,104,10504,10990,622,...,F,PLA3,None,SUP CCZ,GLORIA ESPERANZA MESA REYES,None,CAMILO ANDRES LOPEZ PENAGOS,1024494610,0,1023870284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32613,5/05/2026,URBANO,CE17BG021,1,1,29,105,10282,12852,504160,...,H,None,None,None,None,None,None,0,0,0
32614,8/05/2026,URBANO,CE17BG024,1,1,32,105,10282,12852,504283,...,H,None,None,None,None,None,None,0,0,0
32615,6/05/2026,URBANO,CE12D0036,4,3,36,105,10264,12328,504183,...,H,None,None,None,None,None,None,0,0,0
32616,8/05/2026,URBANO,CE1600008,2,1,8,105,10184,12774,504372,...,H,None,None,None,None,None,None,0,0,0


In [18]:
#Exportar archivo

iri.to_csv(f"Z:/01 base_datos/41 Informe Diario CCZ/2023/IRI TM/regularidad_{mes}.csv", index=False)